# Chapter 12 – Worked examples

This notebook runs the quantitative material of [Chapter 12](../index.md) live. The code lives in the modules
next to it (`two_node_market.py`, `congestion_cost.py`, `smard_api.py`, `smard_week.py`); the cells only call
them, so a change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import congestion_cost
import matplotlib.pyplot as plt
import numpy as np
import smard_api
import smard_week
import two_node_market

## A two-node market with a line limit

Node A: 1,000 MW at 20 €/MWh. Node B: 1,000 MW at 60 €/MWh and 800 MW of load. The line carries at most
500 MW. Nodal pricing clears the market with the limit; a zonal market ignores it and redispatches afterwards.

In [ ]:
n = two_node_market.nodal()
z = two_node_market.zonal()
print('nodal :', n)
print('zonal :', z)

Sweep the line limit and look at the two curves: the redispatch cost falls linearly to zero at the load, the
congestion rent rises linearly and then drops to zero in one step.

In [ ]:
limits = np.linspace(0, 1000, 201)
cost, rent = two_node_market.sweep(limits)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(limits, cost / 1000, label='redispatch cost (zonal market)')
ax.plot(limits, rent / 1000, label='congestion rent (nodal market)')
ax.set_xlabel('line limit [MW]')
ax.set_ylabel('thousand euros per hour')
ax.axvline(two_node_market.LOAD_B, color='grey', linestyle=':')
ax.legend()
plt.show()

## What congestion management costs in Germany

The quarterly SMARD figures: volume of measures, estimated cost and the unit cost that follows.

In [ ]:
table = congestion_cost.QUARTERS.assign(eur_per_mwh=congestion_cost.cost_per_mwh(congestion_cost.QUARTERS))
table.round(2)

In [ ]:
congestion_cost.changes(congestion_cost.QUARTERS).round(2)

## A week of SMARD data

`smard_api.live_or_saved` fetches the week of 12–18 January 2026 from the SMARD JSON API, or reads the copy
saved next to this notebook when the portal cannot be reached. Both have the layout of the download center's
CSV export (semicolon, byte-order mark, `-` for missing, English or German dates and numbers), which
`smard_week.read_smard` parses; the two committed `smard_*_sample.csv` files are a synthetic week in the same
layout, kept for the tests.

In [ ]:
gen, cons, source = smard_api.live_or_saved()
print(source)
gen.head(3)

In [ ]:
residual = smard_week.residual_load(gen, cons)
wind = gen[['Wind offshore', 'Wind onshore']].sum(axis=1)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(cons.index, cons[smard_week.LOAD] / 1000, label='consumption')
ax.plot(wind.index, wind / 1000, label='wind')
ax.plot(residual.index, residual / 1000, label='residual load')
ax.set_ylabel('GW')
ax.legend()
plt.show()

The hours of high wind and low residual load (top quarter of wind, bottom quarter of residual load), and the
daily summary of Exercise 4.

In [ ]:
smard_week.critical_hours(gen, cons).round(0)

In [ ]:
smard_week.daily_summary(gen, cons).round(1)